
# This notebook builds features for hourly price prediction.

In [2]:
import pandas as pd
import numpy as np
import holidays

## 1. Load Data

In [3]:
df = pd.read_csv("finland_electricity_predict_dataset.csv", parse_dates=["datetime"])
df = df.rename(columns={
    "Air temperature mean [°C]": "temp",
    "Wind speed mean [m/s]": "wind_speed",
})
df = df.sort_values("datetime").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["datetime"].min(), "→", df["datetime"].max())
df.head(3)

Shape: (26304, 4)
Date range: 2023-01-01 00:00:00 → 2025-12-31 23:00:00


,datetime,price,temp,wind_speed
0,2023-01-01 00:00:00,4.84,4.433333,8.516667
1,2023-01-01 01:00:00,2.01,4.100000,7.716667
2,2023-01-01 02:00:00,1.38,3.800000,8.150000


## 2. Temporal Features

Basic calendar components extracted from the datetime index.

In [4]:
df["hour"]         = df["datetime"].dt.hour
df["day_of_week"]  = df["datetime"].dt.dayofweek   # 0=Mon, 6=Sun
df["day_of_month"] = df["datetime"].dt.day
df["month"]        = df["datetime"].dt.month
df["week_of_year"] = df["datetime"].dt.isocalendar().week.astype(int)
df["quarter"]      = df["datetime"].dt.quarter
df["year"]         = df["datetime"].dt.year

season_map = {12: 1, 1: 1, 2: 1, 3: 2, 4: 2, 5: 2,
              6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4}
df["season"] = df["month"].map(season_map)

df["is_weekend"]    = (df["day_of_week"] >= 5).astype(int)
df["is_peak_hour"]  = (df["hour"].isin(range(7, 10)) | df["hour"].isin(range(17, 21))).astype(int)
df["is_night_hour"] = (df["hour"].isin(range(23, 24)) | df["hour"].isin(range(0, 6))).astype(int)

df[["datetime","hour","day_of_week","month","season","is_weekend","is_peak_hour","is_night_hour","day_of_month"]]

,datetime,hour,day_of_week,month,season,is_weekend,is_peak_hour,is_night_hour,day_of_month
0,2023-01-01 00:00:00,0,6,1,1,1,0,1,1
1,2023-01-01 01:00:00,1,6,1,1,1,0,1,1
2,2023-01-01 02:00:00,2,6,1,1,1,0,1,1
3,2023-01-01 03:00:00,3,6,1,1,1,0,1,1
4,2023-01-01 04:00:00,4,6,1,1,1,0,1,1
...,...,...,...,...,...,...,...,...,...
26299,2025-12-31 19:00:00,19,2,12,1,0,1,0,31
26300,2025-12-31 20:00:00,20,2,12,1,0,1,0,31
26301,2025-12-31 21:00:00,21,2,12,1,0,0,0,31
26302,2025-12-31 22:00:00,22,2,12,1,0,0,0,31


## 3. Cyclic Encoding

Tree models don't know that hour 23 and hour 0 are adjacent. Sine/cosine transforms fix this.

In [5]:
def cyclic_encode(series, max_val):
    angle = 2 * np.pi * series / max_val
    return np.sin(angle), np.cos(angle)

df["hour_sin"],        df["hour_cos"]        = cyclic_encode(df["hour"], 24)
df["day_of_week_sin"], df["day_of_week_cos"] = cyclic_encode(df["day_of_week"], 7)
df["month_sin"],       df["month_cos"]       = cyclic_encode(df["month"], 12)
df["week_of_year_sin"],df["week_of_year_cos"]= cyclic_encode(df["week_of_year"], 52)

df[["datetime","hour","hour_sin","hour_cos","month","month_sin","month_cos"]].head(6)

,datetime,hour,hour_sin,hour_cos,month,month_sin,month_cos
0,2023-01-01 00:00:00,0,0.000000,1.000000,1,0.5,0.866025
1,2023-01-01 01:00:00,1,0.258819,0.965926,1,0.5,0.866025
2,2023-01-01 02:00:00,2,0.500000,0.866025,1,0.5,0.866025
3,2023-01-01 03:00:00,3,0.707107,0.707107,1,0.5,0.866025
4,2023-01-01 04:00:00,4,0.866025,0.500000,1,0.5,0.866025
5,2023-01-01 05:00:00,5,0.965926,0.258819,1,0.5,0.866025


## 4. Finnish Public Holiday Flag

Holidays cause demand drops similar to weekends. Uses the holidays library for official Finnish holidays.

In [6]:
years = df["datetime"].dt.year.unique().tolist()
fi_holidays = holidays.Finland(years=years)

df["is_holiday"] = df["datetime"].dt.date.astype(str).isin(
    [str(d) for d in fi_holidays.keys()]
).astype(int)

# Combined flag: any non-working hour (weekend or holiday)
df["is_non_working"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)

holiday_counts = df.groupby("is_holiday")["datetime"].count()
print("Non-holiday hours:", holiday_counts.get(0, 0))
print("Holiday hours:    ", holiday_counts.get(1, 0))
df[df["is_holiday"] == 1][["datetime","is_holiday","is_non_working"]].head(8)

Non-holiday hours: 25225
Holiday hours:     1079


,datetime,is_holiday,is_non_working
0,2023-01-01 00:00:00,1,1
1,2023-01-01 01:00:00,1,1
2,2023-01-01 02:00:00,1,1
3,2023-01-01 03:00:00,1,1
4,2023-01-01 04:00:00,1,1
5,2023-01-01 05:00:00,1,1
6,2023-01-01 06:00:00,1,1
7,2023-01-01 07:00:00,1,1


## 5. Lag Features

Lag features give the model access to historical prices. The most important is lag_168h — the same hour exactly one week ago.

In [7]:
for h in [1, 2, 3, 6, 12, 24, 48, 168]:
    df[f"price_lag_{h}h"] = df["price"].shift(h)

df[["datetime", "price", "price_lag_1h", "price_lag_24h", "price_lag_168h"]].head(10)

,datetime,price,price_lag_1h,price_lag_24h,price_lag_168h
0,2023-01-01 00:00:00,4.84,NaN,NaN,NaN
1,2023-01-01 01:00:00,2.01,4.84,NaN,NaN
2,2023-01-01 02:00:00,1.38,2.01,NaN,NaN
3,2023-01-01 03:00:00,0.09,1.38,NaN,NaN
4,2023-01-01 04:00:00,0.08,0.09,NaN,NaN
5,2023-01-01 05:00:00,0.05,0.08,NaN,NaN
6,2023-01-01 06:00:00,0.08,0.05,NaN,NaN
7,2023-01-01 07:00:00,0.09,0.08,NaN,NaN
8,2023-01-01 08:00:00,0.53,0.09,NaN,NaN
9,2023-01-01 09:00:00,2.03,0.53,NaN,NaN


## 6. Rolling Features

Capture recent price trends and volatility over the past 24 hours and 7 days.

In [8]:
# 24-hour rolling stats on price
df["price_rolling_mean_24h"] = df["price"].shift(1).rolling(24).mean()
df["price_rolling_std_24h"]  = df["price"].shift(1).rolling(24).std()
df["price_rolling_min_24h"]  = df["price"].shift(1).rolling(24).min()
df["price_rolling_max_24h"]  = df["price"].shift(1).rolling(24).max()

# 7-day (168h) rolling mean on price
df["price_rolling_mean_168h"] = df["price"].shift(1).rolling(168).mean()

# 24-hour rolling mean on temperature
df["temp_rolling_mean_24h"] = df["temp"].shift(1).rolling(24).mean()

df[["datetime","price","price_rolling_mean_24h","price_rolling_std_24h","price_rolling_mean_168h"]].head(30).tail(6)

,datetime,price,price_rolling_mean_24h,price_rolling_std_24h,price_rolling_mean_168h
24,2023-01-02 00:00:00,35.00,16.977083,20.678492,NaN
25,2023-01-02 01:00:00,57.91,18.233750,20.824751,NaN
26,2023-01-02 02:00:00,51.67,20.562917,22.022923,NaN
27,2023-01-02 03:00:00,52.86,22.658333,22.505553,NaN
28,2023-01-02 04:00:00,44.16,24.857083,22.780885,NaN
29,2023-01-02 05:00:00,50.08,26.693750,22.471257,NaN


## 7. Weather-Derived Features

Proxy features derived from temperature and wind speed that more directly reflect energy supply and demand.

In [9]:
# Heating Degree Hours: how much colder than comfort threshold (17°C)
# Higher value = more heating demand = higher electricity price in Finland
df["HDD"] = (17 - df["temp"]).clip(lower=0)

# Wind power proxy: turbine output scales with wind³, capped at rated speed (13 m/s)
df["wind_power_proxy"] = df["wind_speed"].clip(upper=13) ** 3

# Lagged temperature (yesterday and last week)
df["temp_lag_24h"]  = df["temp"].shift(24)
df["temp_lag_168h"] = df["temp"].shift(168)

df[["datetime","temp","wind_speed","HDD","wind_power_proxy","temp_lag_24h"]].head(8)

,datetime,temp,wind_speed,HDD,wind_power_proxy,temp_lag_24h
0,2023-01-01 00:00:00,4.433333,8.516667,12.566667,617.744588,NaN
1,2023-01-01 01:00:00,4.100000,7.716667,12.900000,459.503921,NaN
2,2023-01-01 02:00:00,3.800000,8.150000,13.200000,541.343375,NaN
3,2023-01-01 03:00:00,3.616667,9.333333,13.383333,813.037037,NaN
4,2023-01-01 04:00:00,2.516667,11.700000,14.483333,1601.613000,NaN
5,2023-01-01 05:00:00,1.883333,13.150000,15.116667,2197.000000,NaN
6,2023-01-01 06:00:00,1.450000,12.833333,15.550000,2113.578704,NaN
7,2023-01-01 07:00:00,0.916667,12.000000,16.083333,1728.000000,NaN


## 8. Save

In [10]:
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist())
print(f"\nNaN counts per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

df.to_csv("finland_electricity_features_v2.csv", index=False)
print("\nSaved → finland_electricity_features_v2.csv")

Total columns: 43
['datetime', 'price', 'temp', 'wind_speed', 'hour', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'year', 'season', 'is_weekend', 'is_peak_hour', 'is_night_hour', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'week_of_year_sin', 'week_of_year_cos', 'is_holiday', 'is_non_working', 'price_lag_1h', 'price_lag_2h', 'price_lag_3h', 'price_lag_6h', 'price_lag_12h', 'price_lag_24h', 'price_lag_48h', 'price_lag_168h', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_min_24h', 'price_rolling_max_24h', 'price_rolling_mean_168h', 'temp_rolling_mean_24h', 'HDD', 'wind_power_proxy', 'temp_lag_24h', 'temp_lag_168h']

NaN counts per column:
price_lag_1h                 1
price_lag_2h                 2
price_lag_3h                 3
price_lag_6h                 6
price_lag_12h               12
price_lag_24h               24
price_lag_48h               48
price_lag_168h             168
price_rolling_mean_2